In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install git+https://github.com/huggingface/transformers.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 13.6 MB/s eta 0:00:0000:01


In [2]:
from transformers import pipeline

pipe = pipeline("text-generation", model="/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1", device_map="auto")
print("Model loaded successfully")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Model loaded successfully


In [3]:
messages = [
    {"role": "user", "content": "Simplify this text for someone with a basic reading level: 'The proliferation of decentralized financial systems has engendered significant volatility in traditional banking paradigms.'"}
]

result = pipe(messages, max_new_tokens=200)
response_text = result[0]["generated_text"][-1]["content"]
print(response_text)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Here are a few ways to simplify that sentence, depending on how simple you need it to be:

**Option 1: Simple and Direct**

> More and more different ways of handling money have made traditional banks very unstable.

**Option 2: Slightly More Detail**

> Because there are now many different, non-bank ways to manage money, traditional banking is becoming very unpredictable.

**Option 3: Very Basic**

> New money systems are making old banks shaky.

***

**Here's a breakdown of the difficult words:**

* **Proliferation:** A lot of growth or spreading. (Means "more and more")
* **Decentralized financial systems:** Money systems that aren't controlled by a single bank or government. (Means "different ways of handling money")
* **Engendered:** Caused or brought about. (Means "made" or "caused")
* **Volatility:** Unpredictability; rapid


In [4]:
def simplify_text(complex_text):
    messages = [
        {"role": "user", "content": f"""Rewrite the following text in simple, easy-to-understand language suitable for someone with a basic reading level.

Rules:
- Give ONLY ONE simplified version, no options or alternatives
- Do not explain word meanings
- Do not add headers, bullet points, or extra commentary
- Just return the simplified sentence(s) directly

Text: {complex_text}"""}
    ]
    result = pipe(messages, max_new_tokens=200)
    return result[0]["generated_text"][-1]["content"].strip()

# Test it
test_output = simplify_text("The proliferation of decentralized financial systems has engendered significant volatility in traditional banking paradigms.")
print(test_output)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


More and more money systems that don't use banks are causing big ups and downs in normal banking.


In [6]:
!pip install gradio -q

In [7]:
import gradio as gr

def app_interface(text):
    return simplify_text(text)

demo = gr.Interface(
    fn=app_interface,
    inputs=gr.Textbox(lines=5, label="Enter complex text", placeholder="Paste a complicated sentence or paragraph here..."),
    outputs=gr.Textbox(lines=5, label="Simplified Version"),
    title="Gemma Text Simplifier",
    description="Rewrites complex text in simple, accessible language using Gemma 4."
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://0f36170db7d114d6c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
